In [1]:
import random


# 不同约束的难度可能会影响实验结果
我们还需要测试单条指令的遵循率
因此可以设计在固定深度d=x的情况下，单独测试每条指令遵循的成功率，以及一些指令组合的遵循成功率。


In [2]:
START_MIN10 = 1      # 起点最小 0.1
START_MAX10 = 99     # 起点最大 9.9
AMAX10 = 500         # 单步操作数最大 50.0

CONSTRAINS = [
'''
The final line of your response MUST be exactly in this format:
####ANSWER=<answer>
Where <answer> is your answer
Do not write anything after the #### line.
''',
'''
The answer MUST be rounded to two decimal place following the rule of rounding off, including cases where the decimal is zero.
''',
'''
The numbers after the decimal point MUST be expressed using English words, one word per digit.
For example, a decimal part of 40 is written as FOUR ZERO, 05 as ZERO FIVE, and 00 as ZERO ZERO.
''',
'''
The English words in the answer MUST be in upper case.
''',
'''
While following other rules, reverse the order of the digits of the integer part (keep the sign and the decimal part unchanged). For example, if the integer part has digits d1 d2 ... dn, output dn ... d2 d1.
''',
'''
The text immediately after ####ANSWER= MUST begin with the exact prefix "RESULT:". For example, write ####ANSWER=RESULT:<answer>, not a separate RESULT: line before ####ANSWER.
''',
'''
The final answer MUST contain no more than 12 whitespace-separated tokens, including the required answer marker.
''',
'''
Replace every occurrence of the digit 0 with Z, every occurrence of the digit 1 with O, and every occurrence of the digit 2 with T. Leave all other characters unchanged.
''',
'''
Append the sum of all decimal digits in the rounded numerical answer as a suffix in square brackets. Do not include the decimal point or the negative sign in the sum.
''',
'''
The first line of the response MUST be 'HELLO WORLD'.
'''
]

def _make_constrain(k):
    '''约束最少为1'''
    if k > len(CONSTRAINS):
        k = len(CONSTRAINS)
    return ''.join(CONSTRAINS[:k])

def _apply(y10, kind, a10):
    if kind == '+': return y10 + a10
    if kind == '-': return y10 - a10

def _draw(rng, n, amax10=AMAX10):
    return [(rng.choice('+-'), rng.randint(1, amax10)) for _ in range(n)]

def make_question_with_d_and_k(rng, d, k, amax10=AMAX10):
    '''串行链：深度 d，约束 k；答案是整条序列（含起点）的最小值'''
    x10 = rng.randint(START_MIN10, START_MAX10)
    ops = _draw(rng, d, amax10)
    y10 = x10
    values10 = [y10]
    constrain = _make_constrain(k)
    for symbol, a10 in ops:
        y10 = _apply(y10, symbol, a10)
        values10.append(y10)
    ans10 = min(values10)
    body = '\n'.join(f'{i+1}. ( {symbol} {a10 / 10:.1f} )' for i, (symbol, a10) in enumerate(ops))
    q = (f'Start with x = {x10 / 10:.1f}. Apply the following operations IN ORDER.'
         f'\n{body}\n'
         f'What is the minimum value of x, including the starting value?')
    return dict(q=q, ans=ans10 / 10, d=d, W=d, cons=constrain)

q = make_question_with_d_and_k(random.Random(42), 10, 5)
print(q['q'])
print(q['ans'])
print(q['cons'])


Start with x = 8.2. Apply the following operations IN ORDER.
1. ( + 1.3 )
2. ( - 12.6 )
3. ( + 7.2 )
4. ( + 34.7 )
5. ( + 30.3 )
6. ( - 1.7 )
7. ( + 4.8 )
8. ( + 12.0 )
9. ( + 28.8 )
10. ( + 36.7 )
What is the minimum value of x, including the starting value?
-3.1

The final line of your response MUST be exactly in this format:
####ANSWER=<answer>
Where <answer> is your answer
Do not write anything after the #### line.

The answer MUST be rounded to two decimal place following the rule of rounding off, including cases where the decimal is zero.

The numbers after the decimal point MUST be expressed using English words, one word per digit.
For example, a decimal part of 40 is written as FOUR ZERO, 05 as ZERO FIVE, and 00 as ZERO ZERO.

The English words in the answer MUST be in upper case.

While following other rules, reverse the order of the digits of the integer part (keep the sign and the decimal part unchanged). For example, if the integer part has digits d1 d2 ... dn, output dn ..

# Generate the whole dataset


In [3]:
random_seed_generator = random.Random(42)
max_d = 20
max_k = 10
q_in_each_grid = 20
q_matrix = []

for d in range(1, max_d + 1):
    k_list = []
    for k in range(1, max_k + 1):
        q_list = []
        for i in range(q_in_each_grid):
            random_seed = random_seed_generator.randint(0, 2**32)
            q = make_question_with_d_and_k(random.Random(random_seed), d, k)
            q_list.append(q)
        k_list.append(q_list)
    q_matrix.append(k_list)

# q_matrix[depth][constrain_num][question_number]
print(q_matrix[1][9][1])
all_answers = [q['ans'] for k_list in q_matrix for q_list in k_list for q in q_list]
print('rows:', len(all_answers))
print('ans range:', min(all_answers), max(all_answers))


{'q': 'Start with x = 4.4. Apply the following operations IN ORDER.\n1. ( + 1.9 )\n2. ( - 46.4 )\nWhat is the minimum value of x, including the starting value?', 'ans': -40.1, 'd': 2, 'W': 2, 'cons': '\nThe final line of your response MUST be exactly in this format:\n####ANSWER=<answer>\nWhere <answer> is your answer\nDo not write anything after the #### line.\n\nThe answer MUST be rounded to two decimal place following the rule of rounding off, including cases where the decimal is zero.\n\nThe numbers after the decimal point MUST be expressed using English words, one word per digit.\nFor example, a decimal part of 40 is written as FOUR ZERO, 05 as ZERO FIVE, and 00 as ZERO ZERO.\n\nThe English words in the answer MUST be in upper case.\n\nWhile following other rules, reverse the order of the digits of the integer part (keep the sign and the decimal part unchanged). For example, if the integer part has digits d1 d2 ... dn, output dn ... d2 d1.\n\nThe text immediately after ####ANSWER= 

In [4]:
import json
from datasets import load_dataset

rows = []
for d, k_list in enumerate(q_matrix):
    for k, q_list in enumerate(k_list):
        for i, q in enumerate(q_list):
            rows.append({
                'd': d + 1,
                'k': k + 1,
                'i': i + 1,
                'q': q['q'],
                'ans': q['ans'],
                'W': q['W'],
                'cons': q['cons']
            })

save_path = 'synthesis_math_dataset.jsonl'

with open(save_path, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

ds = load_dataset('json', data_files=save_path, split='train')
print(ds)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['d', 'k', 'i', 'q', 'ans', 'W', 'cons'],
    num_rows: 4000
})
